In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI
from Math.ti_class import TI_class, VI_class, TR_class
from Math.lm_class import kalman, LinearModel
from Math.accumfeatures import EMA, MA, MSTD, DifferentialEMA, DerivativeEMA
from Strategies.IntensityHawkes_strategy.model_class import HawkesIntensity
from Strategies.IntensityHawkes_strategy.backtest_class import BacktestIB
from Strategies.IntensityHawkes_strategy.strategy_class import StrategyHI, VolumeClass
tol=(1e-1)/2

In [10]:
import sys,os
sys.path.append(r'C:/data/EnergyTrading/Python/')

import cx_Oracle
try:
    cx_Oracle.init_oracle_client(lib_dir=r"C:\Users\andrej\Downloads\instantclient_21_11")
except:
    pass


In [11]:
data_class = TPData()
data_class.create_connection('OracleSQL')
data_class_pg = TPData()
data_class_pg.create_connection('PostgreSQL')
data_class_tp = TPDataDa()

mkt_list = ['de']
tenor_list = ['m']
tn_list = [2]
prod = 'base'
venue_list = ['eex']
# start_date = datetime(2024, 6, 3)
start_date = datetime(2025, 2, 28)
end_date = datetime(2025, 5, 6)

allwd_broker_ids = [1441]

sample_dates = pd.date_range(start_date, end_date, freq='B')

n = 16
n_t = 15
d_t = 1
date_range_dict = {k.date(): pd.date_range(k, periods=n, freq='B')
                   for k in sample_dates[:-n+1]}

n_s = 2

dates = pd.date_range(start_date, end_date, freq='B')
product_date = [dates.shift(1, freq='B') if t == 'da' else
                dates.shift(1, freq='D') if t == 'd' else
                dates.shift(tn, freq='W-MON') if t == 'w' else
                (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
                (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                for t, tn in zip(tenor_list, tn_list)]

start_time = time(9, 0, 0)
end_time = time(18, 0, 0)

tr_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
ba_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
agg_dict = {'price': 'sum', 'volume': 'sum', 'action': 'median',
            'broker_id': 'median', 'count': 'sum'}

for m, t, n, p_dates in zip(mkt_list, tenor_list, tn_list, product_date):
    df_tr, df_ba = pd.DataFrame([]), pd.DataFrame([])
    series = pd.Series(p_dates, index=dates)
    for p_d, ds in series.groupby(series).groups.items():
        bT = datetime.combine(ds[0], start_time)
        eT = datetime.combine(ds[-1], end_time)
        # Trades
        df_tr_aux = data_class.get_trades(m, t, venue_list, p_d, bT, eT, prod)
        # Filter by broker
        if not allwd_broker_ids or t == 'da':
            pass
        else:
            df_tr_aux = df_tr_aux[df_tr_aux['broker_id'].isin(allwd_broker_ids)]
        # Clean trades
        df_ba_aux = data_class_pg.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, None, aonn=False)
        if df_ba_aux.empty:
            df_ba_aux = data_class_tp.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, None, aonn=False)
        df_ba_aux = df_ba_aux.rename(columns={'bidbestprice': 'b_price', 'askbestprice': 'a_price'})
        df_tr_aux = data_class.clean_trades(df_tr_aux, df_ba_aux)
        try:
            df_tr_aux = df_tr_aux.between_time(start_time, end_time)
        except(TypeError):
            pass
        # Group trades
        df_tr_aux['count'] = 1
        #df_tr_aux['price'] *= df_tr_aux['volume']
        #df_tr_aux = df_tr_aux.groupby(df_tr_aux.index).agg(agg_dict)
        #df_tr_aux['price'] /= df_tr_aux['volume']
        df_tr = pd.concat([df_tr, df_tr_aux])
        df_ba = pd.concat([df_ba, df_ba_aux])
        del df_tr_aux, df_ba_aux
    tr_data_dict[m + t + str(n)] = df_tr
    ba_data_dict[m + t + str(n)] = df_ba
    
trades2=pd.concat({k: v for k, v in tr_data_dict.items()}, axis=1)
trades2.columns = ['_'.join([col[-1], col[0]]) for col in trades2.columns]
trades2.index.name='datetime'
trades2=trades2[['price_dem2', 'volume_dem2', 'action_dem2', 'broker_id_dem2']]

ba2=pd.concat({k: v for k, v in ba_data_dict.items()}, axis=1)
ba2.columns = ['bidbestprice_'+ba2.columns[0][0], 'askbestprice_'+ba2.columns[0][0]]

trades2.sort_index(inplace=True)
ba2.sort_index(inplace=True)
ba2.index.name='datetime'
trades2.index.name ='datetime'

products2=[''.join(map(str,(mkt_list+tenor_list+tn_list)))]

ti_inst = TI(trades2, ba2, products2)

ti_inst.prepare_data()


data_raw = ti_inst.data

data = data_raw[~(data_raw['price_dem2'].notnull() & (data_raw['broker_id_dem2'] != 1441))][['price_dem2', 'volume_dem2','bidbestprice_dem2',
                  'askbestprice_dem2', 'mid_dem2', 'trade_side_dem2']].copy()

# data = data_raw[['price_dem2', 'volume_dem2','bidbestprice_dem2',
#                    'askbestprice_dem2', 'mid_dem2', 'trade_side_dem2']].copy()
    
data.columns = [a.split('_')[0] for a in data.columns]
data.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

data.to_csv(r's:\Algo\Files\andrej\Data\int_data_lag_production_sample_dem2_our_db.csv')

Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre


In [12]:
mkt_list = ['de']
tenor_list = ['m']
tn1_list = [2]
tn2_list = []
prod = 'base'
venue_list = ['eex']
start_date = datetime(2025, 2, 28)
end_date = datetime(2025, 5, 6)
n_s = 2
start_date1 = datetime(2025,4,1)
# start_date2 = datetime(2024,10,7)
start_date2 = None

data_class = TPData()
data_class.create_connection('OracleSQL')

trades_instrument = data_class.get_trades_inst(mkt_list[0], venue_list, start_date, end_date,
                                         prod='base', spread_bool=True)

trades_instrument[trades_instrument['broker_id']==1441]

trades_instrument.to_csv(r's:\Algo\Files\andrej\Data\data_instrument_trades_production_sample_dem2_our_db.csv')

Connected to the database oracle
Disconnected from the database oracle
